# 04. XGBoost / LightGBM 모델

## 모델 선택 이유
- 금리·KOSPI 등 **외생변수의 비선형 상호작용** 포착
- 6개 구를 하나의 모델로 학습 → 지역 간 공통 패턴 학습
- **피처 중요도** 분석으로 어떤 변수가 중요한지 설명 가능

## 전략
1. `02_eda.ipynb`에서 생성한 `ml_features.csv` 로드
2. 시계열 교차검증 (TimeSeriesSplit)
3. XGBoost + LightGBM 학습 및 성능 비교
4. 피처 중요도 시각화
5. 5월 25일 예측

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_PROC = Path('../data/processed')
TARGET_DISTRICTS = ['노원구', '은평구', '서대문구', '서초구', '강남구', '송파구']
PREDICT_DATE = '2026-05-25'

ml_df = pd.read_csv(DATA_PROC / 'ml_features.csv', index_col='date', parse_dates=True)
print(f'ML 데이터셋: {ml_df.shape}')
ml_df.head()

## 1. 피처 / 타겟 분리

In [ ]:
TARGET_COL = 'price_index'
EXCLUDE = [TARGET_COL, 'district']

# 수치형 피처만 사용
feature_cols = [c for c in ml_df.columns
                if c not in EXCLUDE and ml_df[c].dtype in [np.float64, np.int64]]
print(f'피처 수: {len(feature_cols)}')
print(feature_cols[:20], '...')

X = ml_df[feature_cols]
y = ml_df[TARGET_COL]

## 2. 시계열 교차검증

In [ ]:
def evaluate_model(model, X, y, n_splits=5):
    """TimeSeriesSplit으로 시계열 교차검증."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    maes, rmses, mapes = [], [], []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
        preds = model.predict(X_val)

        mae  = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mape = np.mean(np.abs((y_val - preds) / y_val)) * 100

        maes.append(mae)
        rmses.append(rmse)
        mapes.append(mape)
        print(f'  Fold {fold+1}: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%')

    return {
        'mae_mean':  np.mean(maes),  'mae_std':  np.std(maes),
        'rmse_mean': np.mean(rmses), 'rmse_std': np.std(rmses),
        'mape_mean': np.mean(mapes), 'mape_std': np.std(mapes),
    }

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    random_state=42,
    tree_method='hist'
)

print('XGBoost 교차검증:')
xgb_scores = evaluate_model(xgb_model, X, y)
print(f'\n평균 MAE={xgb_scores["mae_mean"]:.4f}±{xgb_scores["mae_std"]:.4f}')
print(f'평균 MAPE={xgb_scores["mape_mean"]:.2f}%±{xgb_scores["mape_std"]:.2f}%')

In [ ]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

print('LightGBM 교차검증:')
lgb_scores = evaluate_model(lgb_model, X, y)
print(f'\n평균 MAE={lgb_scores["mae_mean"]:.4f}±{lgb_scores["mae_std"]:.4f}')
print(f'평균 MAPE={lgb_scores["mape_mean"]:.2f}%±{lgb_scores["mape_std"]:.2f}%')

## 3. 전체 데이터로 최종 모델 학습 및 피처 중요도

In [ ]:
# 성능이 더 좋은 모델 선택 (여기서는 XGBoost 최종 학습)
final_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, tree_method='hist'
)
final_model.fit(X, y, verbose=False)

# 피처 중요도 상위 20개
importance = pd.Series(final_model.feature_importances_, index=feature_cols)
top20 = importance.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost 피처 중요도 (상위 20개)', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(DATA_PROC / 'feature_importance.png', dpi=150)
plt.show()

## 4. 5월 25일 예측 (XGBoost / LightGBM)

In [ ]:
def predict_future_ml(model, ml_df, district, steps=3):
    """재귀적 다단계 예측: t+1 예측값을 lag로 사용해 t+2, t+3 예측."""
    district_df = ml_df[ml_df['district'] == district].copy()
    last_row = district_df.iloc[-1].copy()

    predictions = []
    for step in range(1, steps + 1):
        feat = last_row[feature_cols].values.reshape(1, -1)
        pred = float(model.predict(feat)[0])
        predictions.append(pred)

        # 다음 스텝을 위해 lag 업데이트
        last_row['price_lag2']  = last_row.get('price_lag1', pred)
        last_row['price_lag1']  = pred
        if 'price_chg1' in last_row.index and last_row.get('price_lag1') != 0:
            last_row['price_chg1'] = (pred - last_row['price_lag1']) / last_row['price_lag1']

    return predictions


xgb_predictions = {}
lgb_predictions = {}

# LightGBM 전체 학습
lgb_final = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1
)
lgb_final.fit(X, y)

for district in TARGET_DISTRICTS:
    if district not in ml_df['district'].values:
        continue
    xgb_predictions[district] = predict_future_ml(final_model, ml_df, district)
    lgb_predictions[district] = predict_future_ml(lgb_final, ml_df, district)

# 마지막 주(3번째 예측 = 5월 25일 근접)
print(f'\n=== XGBoost / LightGBM 예측 결과 (3주 후) ===')
rows = []
for district in TARGET_DISTRICTS:
    if district not in xgb_predictions:
        continue
    xgb_val = xgb_predictions[district][-1]
    lgb_val  = lgb_predictions[district][-1]
    rows.append({'구': district,
                 'XGBoost': round(xgb_val, 2),
                 'LightGBM': round(lgb_val, 2)})

ml_result_df = pd.DataFrame(rows)
ml_result_df.to_csv(DATA_PROC / 'ml_predictions.csv', index=False)
print(ml_result_df.to_string(index=False))